In [ ]:
import scanpy as sc
from scarches.models.scpoli import scPoli
import scvi
import anndata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import issparse
import scipy.sparse as sparse
from scarches.models.base._utils import _validate_var_names
import sys
import logging
import os
from tqdm import tqdm
from matplotlib import colors

In [ ]:
# load reference
adata_ref = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap.h5ad")

In [ ]:
# replace the current .X with the raw counts from .raw.X
adata_ref.X = adata_ref.raw.X.copy()

adata_ref.raw = None

# subset AnnData object to only the highly variable genes (HVG)
adata_ref = adata_ref[:, adata_ref.var["highly_variable"]].copy()

In [ ]:
# change to gene symbols as var_names
adata_ref.var["ensembl"] = adata_ref.var_names

adata_ref.var_names = adata_ref.var["feature_name"]

In [ ]:
adata_ref.write("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap_pp.h5ad")

In [ ]:
adata_ref = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap/GBmap_pp.h5ad")

In [ ]:
#train reference model
early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
}

scpoli_model = scPoli(
    adata=adata_ref,
    condition_keys="donor_id",
    cell_type_keys='annotation_level_3',
    embedding_dims=10,
    recon_loss='nb',
    )


In [ ]:
scpoli_model.train(
    n_epochs=50,
    pretraining_epochs=40,
    early_stopping_kwargs=early_stopping_kwargs,
    eta=5,
)

In [ ]:
scpoli_model.save("models/ref", overwrite=True)

In [ ]:
scpoli_model = scpoli_model.load(dir_path="models/ref", adata=adata_ref)

In [ ]:
# load query data
adata_query = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/single_cell/preprocessed/sc_merged_annotated.h5ad")

In [ ]:
adata_query.X = adata_query.layers["ambient_rna_removed"]
adata_query.X = np.round(adata_query.X)

In [ ]:
print("Select the varnames for the model....")
varnames_path = "models/ref/var_names.csv"
var_names = np.genfromtxt(varnames_path, delimiter=",", dtype=str)
adata2 = _validate_var_names(adata_query, var_names)

In [ ]:
# Rename the column 'orig.ident' to 'donor_id'
adata2.obs = adata2.obs.rename(columns={"orig.ident": "donor_id"})

# Add column 'annotation_level_3' with all values set to "unknown"
adata2.obs["annotation_level_3"] = "unknown"

In [ ]:
# WITH EMBEDDINGS!

scvi.settings.seed = 42

# Get all unique donor IDs from adata2
unique_donors = adata2.obs['donor_id'].unique()

# Create batches of donor IDs with a batch size of 10 for memory and runtime efficiency
batches = [unique_donors[i:i+10] for i in range(0, len(unique_donors), 10)]

# List to store DataFrames from each batch
result_dfs = []

for donor_batch in tqdm(batches, desc="Processing batches"):
    # Filter adata2 for the current batch of donors
    adata_batch = adata2[adata2.obs['donor_id'].isin(donor_batch)].copy()

    adata_batch.X = adata_batch.X.toarray()
    
    # Load query data for the current batch using your model
    scpoli_query = scPoli.load_query_data(
        adata=adata_batch,
        reference_model=scpoli_model,
        labeled_indices=[],
    )
    
    # Train the query model
    scpoli_query.train(
        n_epochs=50,
        pretraining_epochs=40,
        eta=10
    )
    
    
    # Get latent embeddings from the trained model
    latent_embeddings = scpoli_query.get_latent(adata_batch, mean=True)
    
    # Classify cells to get predictions, probabilities, and uncertainties
    results_dict = scpoli_query.classify(adata_batch, scale_uncertainties=True)
    
    # Extract results 
    level_results = results_dict['annotation_level_3']
    

    batch_data = {
        'barcode': [barcode[:-2] for barcode in adata_batch.obs_names],
        'cell_type_pred': level_results['preds'].tolist(),
        'uncertainty': level_results['uncert'].tolist(),
    }

    # Add each dimension of the embeddings as a new column
    for i in range(latent_embeddings.shape[1]):
        batch_data[f'embedding_{i}'] = latent_embeddings[:, i]
        
    # Convert the dictionary to a DataFrame
    df_batch = pd.DataFrame(batch_data)
    
    result_dfs.append(df_batch)

# Concatenate all DataFrames from each batch
final_df = pd.concat(result_dfs, ignore_index=True)

# Save the combined DataFrame to a CSV file
final_df.to_csv("predictions_with_embeddings.csv", index=False) # goes into scrnaseq.ipynb

print("Processing complete. Results saved to 'predictions_with_embeddings.csv'")